# Rubik AI: training model 4x4 besar (Kaggle atau Colab)

Model 4x4 besar (sekitar 16 juta parameter) dengan scramble training sampai 50 gerakan, supaya AI
mampu menyelesaikan scramble yang lebih panjang dari model laptop (andal sampai sekitar 15 gerakan).

Notebook ini **otomatis mengenali** apakah berjalan di **Kaggle** (utama) atau **Colab** (cadangan).
Training bisa dicicil: setiap sesi berhenti rapi sebelum batas waktu, lalu sesi berikutnya melanjutkan
dari checkpoint terakhir.

## Persiapan di Kaggle (sekali saja)

1. Buat akun di **kaggle.com**, lalu verifikasi nomor telepon (Settings → Phone verification).
   Ini wajib untuk memakai GPU dan internet.
2. Buat token GitHub **baru** (Fine-grained, hanya repository Rubik_AI, Contents: Read-only).
3. Di Kaggle: **Create → New Notebook**, lalu **File → Import Notebook** dan pilih file ini.
4. Panel kanan (**Settings**):
   - **Accelerator: GPU T4 x2** (jangan P100; PyTorch versi baru tidak lagi mendukungnya).
   - **Internet: On**.
5. Menu **Add-ons → Secrets → Add Secret**: Label `GITHUB_TOKEN`, Value = token GitHub Anda,
   lalu centang agar secret itu **terpasang (attached)** ke notebook ini.

## Menjalankan (setiap sesi)

1. Klik **Save Version** → pilih **Save & Run All (Commit)** → **Save**.
   Notebook berjalan di server Kaggle; browser boleh ditutup. Sesi berhenti rapi setelah sekitar 11 jam.
2. Setelah selesai, buka versi itu → tab **Output**: ada `cubenet_4x4_large.pt` dan log-nya.
3. **Sesi berikutnya:** klik **Add Input** → **Your Work** → pilih **output notebook ini** (versi terbaru),
   lalu Save & Run All lagi. Checkpoint otomatis disalin dan training melanjutkan.
4. Saat training mencapai langkah terakhir, sel evaluasi di bawah otomatis berjalan.

Kuota GPU Kaggle gratis sekitar 30 jam per minggu, jadi biasanya cukup 1–2 sesi.

## 1. Lingkungan: Kaggle atau Colab

In [ ]:
import os, glob, shutil, subprocess

ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    SAVE_DIR = "/kaggle/working"          # disimpan sebagai Output versi notebook
    WORKDIR = "/tmp/Rubik_AI"             # kode tidak ikut disimpan ke Output
else:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/Rubik_AI_colab"
    WORKDIR = "/content/Rubik_AI"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = f"{SAVE_DIR}/cubenet_4x4_large.pt"
LOG_PATH = MODEL_PATH.replace(".pt", "_log.csv")

# Kaggle: ambil checkpoint dari Output versi sebelumnya (yang ditambahkan lewat Add Input)
if ON_KAGGLE and not os.path.exists(MODEL_PATH):
    found = sorted(glob.glob("/kaggle/input/**/cubenet_4x4_large.pt", recursive=True))
    if found:
        shutil.copy(found[0], MODEL_PATH)
        prev_log = found[0].replace(".pt", "_log.csv")
        if os.path.exists(prev_log):
            shutil.copy(prev_log, LOG_PATH)
        print("Checkpoint sesi sebelumnya disalin dari", found[0])

print("Platform:", "Kaggle" if ON_KAGGLE else "Colab", "| checkpoint ada:", os.path.exists(MODEL_PATH))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ambil kode dari GitHub

Kaggle membaca token dari **Secrets** (`GITHUB_TOKEN`), sehingga bisa berjalan tanpa diketik.
Colab akan meminta token diketik. Token tidak pernah ditampilkan atau disimpan di notebook.

In [ ]:
REPO = "HaiqalGITuru/Rubik_AI"
if not os.path.exists(WORKDIR):
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        import getpass
        token = getpass.getpass("Tempel GitHub token: ")
    r = subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", WORKDIR],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    del token
os.chdir(WORKDIR)
!ls

## 3. Pengaturan

- Model: 5.000 → 1.000 → 4 residual block (sama dengan model 3x3 besar).
- Setiap langkah: 500 scramble × 50 gerakan = 25.000 posisi.
- **Jangan ubah `STEPS` setelah training dimulai** (jadwal learning rate dihitung dari angka ini).

In [ ]:
STEPS = 200_000
SCRAMBLES = 500
MAX_DEPTH = 50
TIME_LIMIT = 11 if ON_KAGGLE else 0   # jam per sesi (Kaggle maks 12 jam); Colab: 0 = tanpa batas
print(MODEL_PATH)

## 4. Training

Berhenti sendiri dengan rapi setelah `TIME_LIMIT` jam, atau saat mencapai `STEPS`. Checkpoint disimpan
setiap 2.000 langkah.

In [ ]:
!python -m scripts.train --puzzle 4 --device cuda --hidden1 5000 --hidden2 1000 --blocks 4 --scrambles {SCRAMBLES} --max-depth {MAX_DEPTH} --steps {STEPS} --cosine --log-every 500 --eval-every 2000 --time-limit {TIME_LIMIT} --save {MODEL_PATH} --resume

## 5. Grafik training, ekspor, dan evaluasi (hanya jika training sudah selesai)

Kalau training belum mencapai `STEPS`, sel ini hanya membuat grafik dan meminta Anda menjalankan sesi berikutnya.

In [ ]:
import torch, pandas as pd, matplotlib.pyplot as plt

log = pd.read_csv(LOG_PATH).drop_duplicates("step", keep="last")
ev = log.dropna(subset=["solve_d20"])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(log.step, log.loss, color="#2453E8"); ax[0].set_title("Loss"); ax[0].set_xlabel("langkah")
for d, c in [(10, "#A9B3C2"), (15, "#6B85E8"), (20, "#2453E8")]:
    ax[1].plot(ev.step, ev[f"solve_d{d}"] * 100, "o-", color=c, ms=3, label=f"jarak {d}")
ax[1].set_title("Tingkat berhasil (evaluasi ringan)"); ax[1].set_ylabel("%"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/grafik_training_4x4_large.png", dpi=150); plt.show()

step = torch.load(MODEL_PATH, map_location="cpu")["step"]
print(f"Training sampai langkah {step:,} dari {STEPS:,}")
if step < STEPS:
    print("Belum selesai: jalankan sesi berikutnya (Add Input -> output notebook ini -> Save & Run All).")
else:
    INFER = f"{SAVE_DIR}/cubenet_4x4_large_infer.pt"
    !python -m scripts.export_model {MODEL_PATH} {INFER}
    !python -m scripts.eval_4x4 --model {INFER} --device cuda --beams 256 1024 4096 --depths 5 10 15 20 25 30 40 50 --name eval_4x4_large
    shutil.copytree("results/eval_4x4_large", f"{SAVE_DIR}/eval_4x4_large", dirs_exist_ok=True)
    print("Selesai. Unduh dari Output/Drive: cubenet_4x4_large_infer.pt dan folder eval_4x4_large.")

## 6. Membawa hasilnya ke laptop

- `cubenet_4x4_large_infer.pt` → `Rubik_AI/models/`
- folder `eval_4x4_large` → `Rubik_AI/results/` (seret foldernya, jangan "Extract All")
- `cubenet_4x4_large_log.csv` dan `grafik_training_4x4_large.png` → `Rubik_AI/results/colab/`

`jalankan_web.bat` otomatis memakai model 4x4 besar kalau file `_infer`-nya ada.

**Pindah platform di tengah jalan (opsional):** unduh `cubenet_4x4_large.pt` (dan `_log.csv`) dari
Output Kaggle, unggah ke folder `Rubik_AI_colab` di Google Drive, lalu jalankan notebook ini di Colab
(atau sebaliknya: unggah dari Drive ke Kaggle sebagai Dataset, lalu Add Input). Pastikan hanya
**satu** platform yang melatih checkpoint yang sama pada satu waktu.